# Noise Sensors: Drift Detection

Your three nodes were calibrated in September. Since then they run in the network. The reference terminal is still
there for now, but in a real network it would leave.

🕵️ **Your job**: find out for each node **whether** it drifts, **when** it started, **how** (slowly or suddenly)
and **what you would do** about it.

```text
 1 calibrate            2 detect                               3 decide
 September data  ──>    A residuals + control limits    ──>    which node, when,
 one model per node     B node against node (no reference)     how big, what to do
                        C CUSUM change point
                        D an ML anomaly detector
```

---

# 🏆 Challenge

## For each node: drift yes or no, start date, type, size by 28 December, and your recommended action.

📝 Fill in the table in the template. Your tutor knows the true answer and reveals it at the end.

---

# 0 Python Setup and Interface

## 0.0 Notebook Interaction

- **Cells with code**: select the cell and press `Shift`+`Enter` to run it and move on, or `Ctrl`+`Enter` to run it and stay
- **Symbols before code cells**:
  - **▶️ Run**: run the cell as it is
  - **🚀 Action**: run it, change a value, run it again
- \# 💬 **Comment**: Python skips everything after `#`
- Run the cells **from top to bottom**. If something breaks: *Kernel* → *Restart & Run All*.

In [ ]:
# ▶️ Run: get the data (needed on Google Colab, skipped on your own computer)
import os, sys, zipfile
from pathlib import Path

REPO = "euramet_ml_noise_workshop"
REPO_URL = f"https://github.com/kjibran/{REPO}.git"

if not Path("data").exists() and "google.colab" in sys.modules:
    folder = REPO
    if not Path(folder, "data").exists():
        os.system(f"git clone -q {REPO_URL}")
    if not Path(folder, "data").exists():
        from google.colab import files
        print("Download failed. Please upload ml_noise_workshop.zip (from your tutor).")
        zipfile.ZipFile(next(iter(files.upload()))).extractall(".")
        folder = "ml_noise_workshop"
    os.chdir(folder)
print("Data folder found:", Path("data").exists())

In [ ]:
# ▶️ Run
import datetime as dt
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import polars as pl
from sklearn.ensemble import IsolationForest
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

%matplotlib inline
plt.rcParams["figure.figsize"] = (11, 4)
plt.rcParams["date.converter"] = "concise"  # short date labels

## 0.1 Helper Functions

Load, fix the clock, join, as before. No need to read them.

##### ▶️ Run

In [ ]:
# ▶️ Run
FOLDER = Path("./data")
NODES = ["N1", "N2", "N3"]
COLORS = {"N1": "#2a78d6", "N2": "#1baf7a", "N3": "#eb6834", "Reference": "#0b0b0b"}

ref = pl.read_csv(FOLDER / "reference_class1.csv", try_parse_dates=True).rename(
    {"time_utc": "time", "laeq_10min": "ref_laeq"})
meteo = pl.read_csv(FOLDER / "meteo.csv", try_parse_dates=True).rename({"time_utc": "time"})
nodes = pl.concat(
    pl.read_csv(FOLDER / f"node_{node}.csv", try_parse_dates=True).select(pl.lit(node).alias("ID"), pl.all())
    for node in NODES
).sort("ID", "time")


def energy_mean(column: str) -> pl.Expr:
    """Average levels in energy: 10·log10(mean(10^(L/10))). Never take the plain mean of dB values."""
    return 10 * (10 ** (pl.col(column) / 10)).mean().log10()

In [ ]:
# ▶️ Run

def fix_clock(nodes: pl.DataFrame) -> pl.DataFrame:
    """N1 logs local time in Bern (UTC+2 in summer, UTC+1 after 26 Oct). Convert it to UTC like the others.
    The repeated hour on 26 Oct cannot be resolved, so those readings are dropped."""
    n1 = nodes.filter(pl.col("ID") == "N1").with_columns(
        pl.col("time").dt.replace_time_zone("Europe/Zurich", ambiguous="null")
        .dt.convert_time_zone("UTC").dt.replace_time_zone(None)
    ).drop_nulls("time")
    return pl.concat([n1, nodes.filter(pl.col("ID") != "N1")]).sort("ID", "time")

def build_table(nodes: pl.DataFrame) -> pl.DataFrame:
    """One row per node and 10 minutes: node reading, reference, weather. Rows without a reference value are dropped."""
    return (
        fix_clock(nodes)
        .join(ref, on="time", how="left")
        .join(meteo, on="time", how="left")
        .drop_nulls(["laeq_10min", "ref_laeq"])
        .with_columns(
            hour=pl.col("time").dt.hour(),
            day=(pl.col("time") - pl.datetime(2025, 9, 1)).dt.total_minutes() / 1440,
        )
    )

def scores(y_true, y_pred) -> dict:
    """R², RMSE [dB], MBE [dB] and the share of readings within ±2 dB of the reference."""
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    err = y_pred - y_true
    return {
        "R²": round(float(r2_score(y_true, y_pred)), 3),
        "RMSE": round(float(np.sqrt(np.mean(err ** 2))), 2),
        "MBE": round(float(np.mean(err)), 2),
        "within ±2 dB": f"{100 * np.mean(np.abs(err) <= 2):.0f} %",
    }

data = build_table(nodes)
CAL_END = dt.datetime(2025, 9, 29)

## 1 Calibrate Each Node on September

One model per node, fitted on the calibration period only, then applied to the whole period.
The **residual** is `calibrated − reference`. Without drift it stays around zero.

#### 🚀 Action (optional): use the features and model you chose in notebook 01.

In [ ]:
# 🚀 Action
FEATURES = ["laeq_10min", "wind_speed", "rain_mm"]

parts = []
for node in NODES:
    dn = data.filter(pl.col("ID") == node)
    train = dn.filter(pl.col("time") < CAL_END)
    model = LinearRegression().fit(train.select(FEATURES).to_numpy(), train["ref_laeq"].to_numpy())
    parts.append(dn.with_columns(cal=pl.Series(model.predict(dn.select(FEATURES).to_numpy()))))
data = pl.concat(parts).with_columns(residual=pl.col("cal") - pl.col("ref_laeq"))

# daily values, using only periods where the instrument is not masked by weather or the noise floor
quiet_weather = (pl.col("rain_mm") == 0) & (pl.col("wind_speed") < 4) & (pl.col("ref_laeq") > 50)
daily = (data.filter(quiet_weather)
         .group_by("ID", pl.col("time").dt.date().alias("date"))
         .agg(pl.col("residual").median().alias("median_res"), pl.col("residual").std().alias("std_res"),
              pl.corr("cal", "ref_laeq").alias("corr"), pl.col("cal").median().alias("median_cal"), pl.len().alias("n"))
         .filter(pl.col("n") >= 12).sort("ID", "date"))
CAL_DAYS = pl.col("date") < CAL_END.date()
daily.head()

## 2 Detect

### 2.A 🚦 Residuals with Control Limits (needs the reference)

The limits come from the calibration period: mean ± `K` standard deviations of the daily median residual.
An alarm is raised when `CONSECUTIVE` days in a row fall outside.

#### 🚀 Action
- Try `K = 2`, `3`, `4` and `CONSECUTIVE = 1`, `2`, `3`. ❓ What do you gain and lose with tighter limits?

In [ ]:
# 🚀 Action
K = 3
CONSECUTIVE = 2

fig, axes = plt.subplots(3, 1, figsize=(11, 8), sharex=True)
alarms_a = {}
for ax, node in zip(axes, NODES):
    dn = daily.filter(pl.col("ID") == node)
    cal = dn.filter(CAL_DAYS)["median_res"]
    lo, hi = cal.mean() - K * cal.std(), cal.mean() + K * cal.std()
    out = ((dn["median_res"] < lo) | (dn["median_res"] > hi)).to_numpy()
    run = np.convolve(out, np.ones(CONSECUTIVE, int), "valid") == CONSECUTIVE
    first = dn["date"][int(np.argmax(run))] if run.any() else None
    alarms_a[node] = first
    ax.fill_between(dn["date"], lo, hi, color="#D9E2EC")
    ax.plot(dn["date"], dn["median_res"], ".-", color=COLORS[node], lw=1)
    if first: ax.axvline(first, color="#EF3E42", lw=2)
    ax.set_ylabel(f"{node} (dB)")
axes[0].set_title("Daily median residual, calibrated − reference. Shaded: control limits. Red: first alarm")
plt.show()
alarms_a

### 2.B 🤝 Node Against Node (no reference needed)

Co-located nodes hear the same street. Compare each node with the median of all three, day by day.

❓ Which node walks away from the others? What would happen if two of the three nodes drifted the same way?

##### ▶️ Run

In [ ]:
# ▶️ Run
group = daily.group_by("date").agg(pl.col("median_cal").median().alias("group_median"))
peer = daily.join(group, on="date").with_columns(vs_group=pl.col("median_cal") - pl.col("group_median"))
fig, ax = plt.subplots()
for node in NODES:
    dn = peer.filter(pl.col("ID") == node).sort("date")
    ax.plot(dn["date"], dn["vs_group"], ".-", color=COLORS[node], label=node, lw=1)
ax.axhline(0, color="k", lw=0.8)
ax.set_ylabel("Node − median of the three nodes (dB)"); ax.legend()
plt.show()

### 2.C 📐 CUSUM: When Did It Start?

CUSUM adds up small deviations day by day, so a slow drift builds up until it crosses a threshold `H`.
Deviations smaller than the allowance `K_ALLOW` are ignored. The **start** estimate is the last day the sum was zero
before the alarm.

#### 🚀 Action
- Try `K_ALLOW = 0.25`, `0.5`, `1.0` and `H = 2`, `4`, `8` (dB). ❓ Which setting finds the slow drift earliest without false alarms on N1?

In [ ]:
# 🚀 Action
K_ALLOW = 0.5
H = 4.0

rows = []
for node in NODES:
    dn = daily.filter(pl.col("ID") == node)
    x = dn["median_res"].to_numpy() - dn.filter(CAL_DAYS)["median_res"].mean()
    s_up, s_down, alarm, start = 0.0, 0.0, None, None
    last_zero = 0
    for i, v in enumerate(x):
        s_up, s_down = max(0, s_up + v - K_ALLOW), max(0, s_down - v - K_ALLOW)
        if s_up == 0 and s_down == 0:
            last_zero = i
        if alarm is None and (s_up > H or s_down > H):
            alarm, start = i, last_zero + 1
            break
    rows.append({"node": node, "alarm date": dn["date"][alarm] if alarm is not None else None,
                 "estimated start": dn["date"][start] if start is not None else None,
                 "direction": None if alarm is None else ("up" if s_up > H else "down")})
pl.DataFrame(rows)

### 2.D 🤖 An ML Anomaly Detector

An **Isolation Forest** learns what a normal day looks like from the calibration period: daily median residual,
scatter of the residual and correlation with the reference. Each later day gets a score, and days that look unlike
September are flagged.

#### 🚀 Action
- Change `CONTAMINATION` (the share of September days allowed to look odd): `0.01`, `0.05`, `0.1`.
- Remove `"median_res"` from `DAY_FEATURES`. ❓ Can it still see the drift? Which feature carries the information?

In [ ]:
# 🚀 Action
CONTAMINATION = 0.05
DAY_FEATURES = ["median_res", "std_res", "corr"]

fig, axes = plt.subplots(3, 1, figsize=(11, 8), sharex=True)
for ax, node in zip(axes, NODES):
    dn = daily.filter(pl.col("ID") == node)
    iso = IsolationForest(contamination=CONTAMINATION, random_state=0).fit(dn.filter(CAL_DAYS).select(DAY_FEATURES).to_numpy())
    flag = iso.predict(dn.select(DAY_FEATURES).to_numpy()) == -1
    ax.plot(dn["date"], dn["median_res"], "-", color="#9FA1A4", lw=1)
    ax.scatter(dn["date"].filter(pl.Series(flag)), dn["median_res"].filter(pl.Series(flag)), color="#EF3E42", s=18,
               label="flagged day", zorder=3)
    ax.set_ylabel(f"{node} (dB)")
axes[0].legend(); axes[0].set_title("Daily median residual, red: days flagged as anomalous")
plt.show()

## 3 Decide

### 3.1 ⏳ When Would It Leave the Tolerance?

Fit a straight line to the daily median residual after an estimated start date. The slope is the drift rate,
and from it you get the days until a tolerance is exceeded.

#### 🚀 Action
- Set `NODE`, `START` (from your CUSUM result) and `TOLERANCE_DB`.

In [ ]:
# 🚀 Action
NODE = "N2"
START = dt.date(2025, 10, 1)
TOLERANCE_DB = 1.0

dn = daily.filter((pl.col("ID") == NODE) & (pl.col("date") >= START))
days = np.array([(x - START).days for x in dn["date"]])
slope, intercept = np.polyfit(days, dn["median_res"].to_numpy(), 1)
print(f"{NODE}: drift rate {slope:+.3f} dB per day, {30 * slope:+.2f} dB per month")
if slope != 0:
    print(f"Reaches {TOLERANCE_DB} dB about {abs(TOLERANCE_DB / slope):.0f} days after {START}")

---

# 4 📝 Wrap-up

Fill in the table in the template:

| Node | Drift? | Type | Start date | Size by 28 Dec | Found first by | Action |
|---|---|---|---|---|---|---|
| N1 | | | | | | |
| N2 | | | | | | |
| N3 | | | | | | |

- 🔭 **Methods**: which method found which fault first? Which one needs the reference?
- 🗓️ **Recalibration**: how often would you recalibrate a node like N2? What decides the interval?
- 🤔 **Open question**: how would you detect drift in a network where no node is next to a reference?